# AAhar food detector — Run All

1. **Runtime → Change runtime type → T4 GPU**
2. **Left sidebar → key icon → Add secret** `HF_TOKEN` (your Hugging Face write token)
3. Edit the **Config** cell (dataset source + HF repo name)
4. **Runtime → Run all**

Trains YOLOv11m on your food dataset and pushes `best.pt` + ONNX + TFLite to your HF model repo.

## 1. Install

In [ ]:
!pip install -q ultralytics==8.3.* huggingface_hub==0.26.* onnx onnxruntime 2>&1 | tail -3
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Config — edit these

In [ ]:
REPO_URL      = 'https://github.com/workspace220106/Aahar-Kitchen-System.git'
REPO_BRANCH   = 'train-detector'

# Pick ONE dataset source. Images must be a flat folder-per-class tree OR already YOLO-formatted.
# Examples:
#   DATASET_SOURCE = 'hf://your-username/aahar-images'
#   DATASET_SOURCE = 'gdrive://1AbC...xyz'        # folder id
#   DATASET_SOURCE = 'url://https://example.com/aahar-dataset.zip'
DATASET_SOURCE = 'url://REPLACE_WITH_YOUR_DATASET_ZIP'

HF_MODEL_REPO = 'your-username/aahar-detector'   # will be created if missing

MODEL         = 'yolo11m.pt'   # m = ~20M params, fits T4; use yolo11s.pt if OOM
IMG_SIZE      = 640
EPOCHS        = 80
BATCH         = 16
VAL_SPLIT     = 0.1            # used only when DATASET_SOURCE is folder-per-class

## 3. Clone repo + fetch dataset

In [ ]:
import os, subprocess, shutil, zipfile, urllib.request, random, pathlib
from pathlib import Path

os.chdir('/content')
if not Path('/content/aahar').exists():
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,'/content/aahar'], check=True)
os.chdir('/content/aahar')

RAW = Path('/content/raw_dataset')
RAW.mkdir(exist_ok=True)

scheme, _, loc = DATASET_SOURCE.partition('://')
if scheme == 'url':
    zip_path = RAW / 'dataset.zip'
    print('Downloading', loc)
    urllib.request.urlretrieve(loc, zip_path)
    with zipfile.ZipFile(zip_path) as z: z.extractall(RAW)
    zip_path.unlink()
elif scheme == 'hf':
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=loc, repo_type='dataset', local_dir=str(RAW))
elif scheme == 'gdrive':
    from google.colab import drive
    drive.mount('/content/drive')
    src = Path('/content/drive/MyDrive') / loc
    shutil.copytree(src, RAW, dirs_exist_ok=True)
else:
    raise ValueError(f'Unknown DATASET_SOURCE scheme: {scheme}')

print('Raw dataset contents:')
for p in list(RAW.rglob('*'))[:20]: print(' ', p.relative_to(RAW))

## 4. Normalize to YOLO layout

If the raw dataset is already in `images/{train,val}` + `labels/{train,val}` form, we keep it.
Otherwise we assume **folder-per-class classification** and convert each image to a single
full-frame bounding box (class 0..N–1 from folder name). That's the right default for
food-ID datasets scraped as class folders — the detector learns to localize the dish
while classifying it.

In [ ]:
import yaml
from PIL import Image

DATA = Path('/content/aahar/data')
for sub in ('images/train','images/val','labels/train','labels/val'):
    (DATA / sub).mkdir(parents=True, exist_ok=True)

already_yolo = (RAW / 'images' / 'train').is_dir() and (RAW / 'labels' / 'train').is_dir()

if already_yolo:
    for sub in ('images/train','images/val','labels/train','labels/val'):
        src = RAW / sub
        if src.is_dir():
            for f in src.iterdir():
                shutil.copy2(f, DATA / sub / f.name)
    names_path = RAW / 'classes.txt'
    class_names = names_path.read_text().splitlines() if names_path.exists() else []
else:
    # Find the deepest level that contains class folders of images.
    roots = [d for d in RAW.iterdir() if d.is_dir()]
    src_root = roots[0] if len(roots) == 1 else RAW
    class_dirs = sorted([d for d in src_root.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_dirs]
    random.seed(42)
    for cid, d in enumerate(class_dirs):
        imgs = [p for p in d.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
        random.shuffle(imgs)
        n_val = max(1, int(len(imgs) * VAL_SPLIT)) if len(imgs) > 1 else 0
        for i, img in enumerate(imgs):
            split = 'val' if i < n_val else 'train'
            try:
                with Image.open(img) as im:
                    im.verify()
            except Exception:
                continue
            dst_img = DATA / f'images/{split}' / f'{cid:04d}_{img.name}'
            dst_lbl = DATA / f'labels/{split}' / f'{cid:04d}_{img.stem}.txt'
            shutil.copy2(img, dst_img)
            dst_lbl.write_text(f'{cid} 0.5 0.5 1.0 1.0\n')

cfg_path = Path('/content/aahar/aahar.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg['path'] = str(DATA)
cfg['names'] = class_names
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(f'{len(class_names)} classes')
print(f'train imgs: {len(list((DATA/"images/train").iterdir()))}')
print(f'val   imgs: {len(list((DATA/"images/val").iterdir()))}')

## 5. Train

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=str(cfg_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    project='/content/runs',
    name='aahar',
    patience=20,
    plots=True,
)
best = Path(results.save_dir) / 'weights' / 'best.pt'
print('best:', best, 'exists:', best.exists())

## 6. Export ONNX + TFLite

In [ ]:
trained = YOLO(str(best))
onnx_path   = trained.export(format='onnx',   imgsz=IMG_SIZE, opset=12, dynamic=False, simplify=True)
tflite_path = trained.export(format='tflite', imgsz=IMG_SIZE, int8=False)
print('onnx:',   onnx_path)
print('tflite:', tflite_path)

## 7. Push to Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, create_repo

token = userdata.get('HF_TOKEN')
assert token, 'Add HF_TOKEN in Colab secrets (left sidebar → key icon)'

api = HfApi(token=token)
create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True, token=token)

for f in [best, Path(onnx_path), Path(tflite_path), cfg_path]:
    api.upload_file(path_or_fileobj=str(f), path_in_repo=f.name, repo_id=HF_MODEL_REPO, repo_type='model')
    print('uploaded', f.name)

print(f'\n→ https://huggingface.co/{HF_MODEL_REPO}')